# 05b - Supporting Predictive Analysis: Fare Amount Regression

This notebook implements the proposal's optional regression analysis:
- **Linear Regression** baseline model
- **Decision Tree Regressor** to capture non-linear pricing effects

Target: `fare_amount`

Evaluation metrics:
- RMSE
- R²

In [0]:

from typing import List, Dict, Tuple

from pyspark.sql import DataFrame
from pyspark.sql import functions as SQL_FUNCTIONS

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:

# Input (Silver)
SILVER_TABLE = "workspace.bda_taxi.taxi_silver"

# Output tables
REG_METRICS_TABLE = "workspace.bda_taxi.fare_regression_metrics"
LR_PREDICTIONS_TABLE = "workspace.bda_taxi.fare_regression_predictions_lr"
DT_PREDICTIONS_TABLE = "workspace.bda_taxi.fare_regression_predictions_dt"

# Train/test split
TRAIN_RATIO = 0.8
SEED = 42

# Saving only a sample of predictions to keep tables small
SAVE_PREDICTION_SAMPLE = True
PREDICTION_SAMPLE_N = 200000

print("SILVER_TABLE:", SILVER_TABLE)
print("REG_METRICS_TABLE:", REG_METRICS_TABLE)
print("LR_PREDICTIONS_TABLE:", LR_PREDICTIONS_TABLE)
print("DT_PREDICTIONS_TABLE:", DT_PREDICTIONS_TABLE)

##### Helper functions

In [0]:

def ensure_schema_exists_for_table(table_fqn: str) -> None:
    parts = table_fqn.split(".")
    if len(parts) < 3:
        raise ValueError(f"Expected catalog.schema.table, got: {table_fqn}")
    schema_fqn = ".".join(parts[:2])
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_fqn}")

def write_delta_table_overwrite(input_dataframe: DataFrame, target_table_fqn: str) -> None:
    ensure_schema_exists_for_table(target_table_fqn)
    (
        input_dataframe.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(target_table_fqn)
    )

def pick_existing_columns(df: DataFrame, candidates: List[str]) -> List[str]:
    existing = set(df.columns)
    return [c for c in candidates if c in existing]

def build_regression_preprocess_pipeline(
    numeric_cols: List[str],
    categorical_cols: List[str],
    features_col: str = "features"
) -> Tuple[List, List[str]]:
    """
    Returns:
    - list of preprocessing stages (indexers + encoder + assembler)
    - list of final input cols used by assembler (numeric + ohe outputs)
    """
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in categorical_cols
    ]

    ohe_input_cols = [f"{c}_idx" for c in categorical_cols]
    ohe_output_cols = [f"{c}_ohe" for c in categorical_cols]

    encoder = OneHotEncoder(
        inputCols=ohe_input_cols,
        outputCols=ohe_output_cols,
        handleInvalid="keep"
    )

    assembler_inputs = list(numeric_cols) + ohe_output_cols

    assembler = VectorAssembler(
        inputCols=assembler_inputs,
        outputCol=features_col,
        handleInvalid="keep"
    )

    stages = indexers + [encoder, assembler]
    return stages, assembler_inputs

def evaluate_regression(predictions_df: DataFrame, label_col: str, prediction_col: str = "prediction") -> Dict[str, float]:
    rmse_eval = RegressionEvaluator(labelCol=label_col, predictionCol=prediction_col, metricName="rmse")
    r2_eval = RegressionEvaluator(labelCol=label_col, predictionCol=prediction_col, metricName="r2")
    mae_eval = RegressionEvaluator(labelCol=label_col, predictionCol=prediction_col, metricName="mae")

    return {
        "rmse": float(rmse_eval.evaluate(predictions_df)),
        "r2": float(r2_eval.evaluate(predictions_df)),
        "mae": float(mae_eval.evaluate(predictions_df))
    }

##### Load and prepare regression dataset

In [0]:
silver_df = spark.table(SILVER_TABLE)

# Target
LABEL_COL = "fare_amount"

# Basic validity filters for fare regression
regression_df = (
    silver_df
    .filter(SQL_FUNCTIONS.col(LABEL_COL).isNotNull())
    .filter(SQL_FUNCTIONS.col("trip_distance").isNotNull())
    .filter(SQL_FUNCTIONS.col("trip_distance") > 0)
    .filter(SQL_FUNCTIONS.col(LABEL_COL) >= 0)
)

print("Regression rows:", regression_df.count())
display(regression_df.select(LABEL_COL, "trip_distance", "PU_Zone", "DO_Zone").limit(10))

##### Feature seletion (zone based)

In [0]:

# Numeric features (avoid leakage like total_amount / tip_amount)
numeric_feature_candidates = [
    "trip_distance",
    "passenger_count"
]


# Mainly add-ons rather than base fare
optional_numeric_candidates = [
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee"
]

# Categorical features (zone-based for consistency with your other work)
categorical_feature_candidates = [
    "PU_Zone",
    "DO_Zone",
    "pickup_hour",
    "pickup_dow",
    "time_bucket",
    "RatecodeID"
]

numeric_features = pick_existing_columns(regression_df, numeric_feature_candidates)
numeric_features += pick_existing_columns(regression_df, optional_numeric_candidates)

categorical_features = pick_existing_columns(regression_df, categorical_feature_candidates)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

##### Trand and test split (80:20)

In [0]:
train_df, test_df = regression_df.randomSplit([TRAIN_RATIO, 1 - TRAIN_RATIO], seed=SEED)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

##### Train models (Linear Regression and Decision Tree)

In [0]:
# Preprocessing stages
preprocess_stages, assembler_inputs = build_regression_preprocess_pipeline(
    numeric_cols=numeric_features,
    categorical_cols=categorical_features,
    features_col="features"
)

# Model 1: Linear Regression baseline
lr_reg = LinearRegression(
    featuresCol="features",
    labelCol=LABEL_COL,
    predictionCol="prediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=preprocess_stages + [lr_reg])
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

print("Trained Linear Regression model.")

# Model 2: Decision Tree Regressor (non-linear)
dt_reg = DecisionTreeRegressor(
    featuresCol="features",
    labelCol=LABEL_COL,
    predictionCol="prediction",
    maxDepth=10,
    # Need it this high because PU_Zone/DO_Zone have so many categories after indexing
    maxBins=512,
    seed=SEED
)

dt_pipeline = Pipeline(stages=preprocess_stages + [dt_reg])
dt_model = dt_pipeline.fit(train_df)
dt_predictions = dt_model.transform(test_df)

print("Trained Decision Tree Regressor model.")

##### Evaluate and compare modesl (RMSE, R^2)

In [0]:

lr_metrics = evaluate_regression(lr_predictions, LABEL_COL)
dt_metrics = evaluate_regression(dt_predictions, LABEL_COL)

print("Linear Regression metrics:", lr_metrics)
print("Decision Tree metrics:", dt_metrics)

comparison_df = spark.createDataFrame([
    ("LinearRegression", lr_metrics["rmse"], lr_metrics["r2"], lr_metrics["mae"]),
    ("DecisionTreeRegressor", dt_metrics["rmse"], dt_metrics["r2"], dt_metrics["mae"])
], ["model", "rmse", "r2", "mae"])

comparison_df = comparison_df.withColumn("run_ts", SQL_FUNCTIONS.current_timestamp())

display(comparison_df.orderBy("rmse"))

write_delta_table_overwrite(comparison_df, REG_METRICS_TABLE)
print("Saved regression metrics table:", REG_METRICS_TABLE)

##### Persist samples

In [0]:

if SAVE_PREDICTION_SAMPLE:
    columns_to_keep = pick_existing_columns(
        lr_predictions,
        [
            LABEL_COL,
            "prediction",
            "trip_distance",
            "passenger_count",
            "PU_Zone",
            "DO_Zone",
            "pickup_hour",
            "pickup_dow",
            "time_bucket",
            "RatecodeID"
        ]
    )

    lr_out = (
        lr_predictions
        .select(*columns_to_keep)
        .withColumn("model_name", SQL_FUNCTIONS.lit("LinearRegression"))
        .orderBy(SQL_FUNCTIONS.rand(SEED))
        .limit(PREDICTION_SAMPLE_N)
    )

    dt_out = (
        dt_predictions
        .select(*columns_to_keep)
        .withColumn("model_name", SQL_FUNCTIONS.lit("DecisionTreeRegressor"))
        .orderBy(SQL_FUNCTIONS.rand(SEED))
        .limit(PREDICTION_SAMPLE_N)
    )

    write_delta_table_overwrite(lr_out, LR_PREDICTIONS_TABLE)
    write_delta_table_overwrite(dt_out, DT_PREDICTIONS_TABLE)

    print("Saved LR predictions sample:", LR_PREDICTIONS_TABLE)
    print("Saved DT predictions sample:", DT_PREDICTIONS_TABLE)
else:
    print("Skipping prediction persistence (SAVE_PREDICTION_SAMPLE=False).")


### Conclusions (Supporting Regression)
- The **Linear Regression** model acts as a baseline to estimate fare amounts under an assumption of linear relationships.
- The **Decision Tree Regressor** can capture non-linear interactions (e.g., zone + time + distance effects).
- We evaluate using:
   - **RMSE**: typical prediction error magnitude (in $)
   - **R²**: proportion of variance explained
- The results are saved to `workspace.bda_taxi.fare_regression_metrics` for inclusion in the report.

**Notes:**
- Using `PU_Zone`/`DO_Zone` provides consistency with earlier analysis but increases categorical cardinality.
- We set `maxBins=512` on the tree to support high-cardinality categorical features.